# Exploratory Data Analysis (EDA) with Polars

This notebook provides a practical guide to performing EDA using Polars, a high-performance DataFrame library.

## 1. Installation and Setup

In [ ]:
!pip install polars matplotlib seaborn

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

## 2. Creating Sample Data

Let's create a sample dataset to work with:

In [ ]:
# Create a sample DataFrame
df = pl.DataFrame({
    'id': range(1000),
    'age': np.random.randint(18, 65, 1000),
    'income': np.random.normal(50000, 15000, 1000),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], 1000),
    'experience': np.random.randint(0, 30, 1000),
    'date': pl.date_range(
        start=pl.date(2020, 1, 1),
        end=pl.date(2023, 12, 31),
        interval='1d',
        eager=True
    )[:1000]
})

# Add some missing values
df = df.with_columns(
    pl.when(pl.col('id') % 10 == 0)
    .then(None)
    .otherwise(pl.col('income'))
    .alias('income')
)

df.head()

## 3. Basic DataFrame Operations

In [ ]:
# Get DataFrame shape
print(f"Shape: {df.shape}")

# Get column names
print(f"Columns: {df.columns}")

# Get data types
print(f"Data types: {df.dtypes}")

# Get summary statistics
df.describe()

## 4. Data Overview

In [ ]:
# Check for missing values
print("Missing values:")
df.null_count()

In [ ]:
# Get unique values in education column
print("Unique education levels:")
df['education'].unique()

In [ ]:
# Get value counts for education
print("Education distribution:")
df['education'].value_counts()

## 5. Statistical Analysis

In [ ]:
# Basic statistics for numeric columns
numeric_stats = df.select([
    pl.col('age').mean().alias('mean_age'),
    pl.col('age').median().alias('median_age'),
    pl.col('age').std().alias('std_age'),
    pl.col('income').mean().alias('mean_income'),
    pl.col('income').median().alias('median_income'),
    pl.col('income').std().alias('std_income')
])

numeric_stats

In [ ]:
# Correlation between age and income
correlation = df.select(
    pl.corr('age', 'income').alias('age_income_correlation')
)
correlation

## 6. Data Visualization

In [ ]:
# Convert to pandas for visualization
df_pd = df.to_pandas()

# Set style
sns.set_style('whitegrid')

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Age distribution
sns.histplot(data=df_pd, x='age', ax=axes[0, 0])
axes[0, 0].set_title('Age Distribution')

# Income distribution
sns.histplot(data=df_pd, x='income', ax=axes[0, 1])
axes[0, 1].set_title('Income Distribution')

# Education distribution
sns.countplot(data=df_pd, x='education', ax=axes[1, 0])
axes[1, 0].set_title('Education Distribution')
axes[1, 0].tick_params(axis='x', rotation=45)

# Age vs Income scatter plot
sns.scatterplot(data=df_pd, x='age', y='income', ax=axes[1, 1])
axes[1, 1].set_title('Age vs Income')

plt.tight_layout()
plt.show()

## 7. Data Cleaning

In [ ]:
# Fill missing values with median
income_median = df['income'].median()
df_cleaned = df.fill_null(income_median)

# Verify no more missing values
print("Missing values after cleaning:")
df_cleaned.null_count()

## 8. Grouping and Aggregation

In [ ]:
# Group by education and calculate statistics
education_stats = df_cleaned.groupby('education').agg([
    pl.col('income').mean().alias('mean_income'),
    pl.col('income').median().alias('median_income'),
    pl.col('age').mean().alias('mean_age'),
    pl.count().alias('count')
]).sort('mean_income', descending=True)

education_stats

## 9. Time Series Analysis

In [ ]:
# Group by month and calculate average income
monthly_stats = df_cleaned.groupby_dynamic(
    'date',
    every='1mo'
).agg([
    pl.col('income').mean().alias('mean_income'),
    pl.col('income').count().alias('count')
])

monthly_stats

## 10. Feature Engineering

In [ ]:
# Create new features
df_enhanced = df_cleaned.with_columns([
    (pl.col('income') / pl.col('age')).alias('income_per_age'),
    (pl.col('experience') / pl.col('age')).alias('experience_ratio'),
    pl.col('income').log().alias('log_income')
])

df_enhanced.head()

## 11. Data Quality Checks

In [ ]:
# Check for outliers in income using IQR
q1 = df_cleaned['income'].quantile(0.25)
q3 = df_cleaned['income'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df_cleaned.filter(
    (pl.col('income') < lower_bound) |
    (pl.col('income') > upper_bound)
)

print(f"Number of outliers: {len(outliers)}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")

## 12. Complete EDA Workflow

In [ ]:
def perform_eda(df):
    # 1. Initial data overview
    print("Data Overview:")
    print(df.head())
    print("\nShape:", df.shape)
    print("\nColumns:", df.columns)
    print("\nData types:", df.dtypes)
    
    # 2. Check for missing values
    print("\nMissing Values:")
    print(df.null_count())
    
    # 3. Basic statistics
    print("\nBasic Statistics:")
    print(df.describe())
    
    # 4. Data quality checks
    print("\nData Quality Checks:")
    for col in df.columns:
        if df[col].dtype in [pl.Float64, pl.Int64]:
            print(f"\n{col} statistics:")
            print(df[col].describe())
            
    # 5. Value counts for categorical columns
    print("\nCategorical Value Counts:")
    for col in df.columns:
        if df[col].dtype == pl.Utf8:
            print(f"\n{col} value counts:")
            print(df[col].value_counts())

# Run the EDA workflow
perform_eda(df_cleaned)